# Indicadores gerais (exploratório)

**Objetivo:** contar os CNPJs ativos, abertos e extintos no ano de referência, separando matrizes e filiais.

**Entrada:** `df_cnpj_espacial` (notebook 08). **Saída:** só a tabela na tela.

**Fora do fluxo:** não entra no `executar_tudo.py` e não é entrada de nenhuma etapa. Para mudar o ano, altere `ANO_REFERENCIA` na célula de cálculo.

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("exploratorio_indicadores_gerais", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== ENTRADA (08) =====
df_cnpj_espacial = fontes.ler_processamento(cfg, "df_cnpj_espacial")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Indicadores
Ativas (situação 02), abertas no ano (data de início) e extintas no ano (situação 08 com data da situação no ano).

In [ ]:
# ============================================================
# Indicadores gerais (Ativas, Abertas e Extintas no ano de referência)
# Usa o DataFrame final: df_cnpj_espacial
# ============================================================

ANO_REFERENCIA = "2026"  # Ano usado nos indicadores "Abertas" e "Extintas". Para mudar o ano, altere só aqui.

log.info("Iniciando o cálculo dos indicadores gerais...")

# ------------------------------------------------------------
# ETAPA 1 – Preparar as colunas (tudo como texto, sem espaços)
# ------------------------------------------------------------
log.info("Etapa 1: preparando as colunas de matriz/filial, situação e datas...")

# Matriz/filial: vira texto e tira espaços (assim "1", 1 e " 1 " ficam iguais)
matriz_filial = df_cnpj_espacial["matriz_filial"].astype(str).str.strip()

# Situação cadastral: vira texto e completa com zero à esquerda ("2" vira "02")
situacao = df_cnpj_espacial["situacao_cadastral"].astype(str).str.strip().str.zfill(2)

# Datas: viram texto (formato AAAAMMDD); o ano são os 4 primeiros caracteres
data_inicio = df_cnpj_espacial["data_inicio_atividade"].astype(str).str.strip()
data_situacao = df_cnpj_espacial["data_situacao"].astype(str).str.strip()

log.info("Etapa 1 concluída!")

# ------------------------------------------------------------
# ETAPA 2 – Criar os filtros (True/False para cada CNPJ)
# ------------------------------------------------------------
log.info("Etapa 2: criando os filtros de cada indicador...")

# Tipo do estabelecimento: 1 = matriz, 2 = filial
eh_matriz = matriz_filial == "1"
eh_filial = matriz_filial == "2"

# Ativa: situação cadastral 02, valendo a situação atual da base
eh_ativa = situacao == "02"

# Aberta no ano: a data de início da atividade começa com o ano (ex.: "2026...")
# Conta mesmo que hoje a empresa já esteja baixada
eh_aberta_no_ano = data_inicio.str[:4] == ANO_REFERENCIA

# Extinta no ano: situação 08 (baixada) E a data da situação é do ano
eh_extinta_no_ano = (situacao == "08") & (data_situacao.str[:4] == ANO_REFERENCIA)

log.info("Etapa 2 concluída!")

# ------------------------------------------------------------
# ETAPA 3 – Conferência: todo CNPJ deve ser matriz ou filial
# ------------------------------------------------------------
log.info("Etapa 3: conferindo se todos os CNPJs são matriz ou filial...")

# Conta os CNPJs que não são nem matriz nem filial (o esperado é zero)
nao_classificados = int((~(eh_matriz | eh_filial)).sum())
if nao_classificados > 0:
    log.warning(f"ATENÇÃO: {nao_classificados} CNPJs com código de matriz/filial diferente de 1 e 2!")
else:
    log.info("Etapa 3 concluída! Todos os CNPJs são matriz ou filial.")

# ------------------------------------------------------------
# ETAPA 4 – Calcular os 9 números (Total, Matrizes, Filiais)
# ------------------------------------------------------------
log.info("Etapa 4: contando os CNPJs de cada indicador...")

def contar(filtro):
    """Recebe um filtro e devolve (total, matrizes, filiais)."""
    matrizes = int((filtro & eh_matriz).sum())  # CNPJs do filtro que são matriz
    filiais = int((filtro & eh_filial).sum())   # CNPJs do filtro que são filial
    total = matrizes + filiais                  # total = matrizes + filiais
    return total, matrizes, filiais

indicadores = {
    "Ativas": contar(eh_ativa),
    f"Abertas em {ANO_REFERENCIA}": contar(eh_aberta_no_ano),
    f"Extintas em {ANO_REFERENCIA}": contar(eh_extinta_no_ano),
}

log.info("Etapa 4 concluída!")

# ------------------------------------------------------------
# ETAPA 5 – Mostrar os resultados (milhar com ponto: 118.783)
# ------------------------------------------------------------
log.info("Etapa 5: montando a tabela de resultados...")

def formatar(numero):
    """Troca a vírgula do milhar por ponto (118,783 vira 118.783)."""
    return f"{numero:,}".replace(",", ".")

# Tabela com uma linha por indicador e uma coluna para cada tipo
df_indicadores = pd.DataFrame(
    {nome: [formatar(v) for v in valores] for nome, valores in indicadores.items()},
    index=["Total de Empresas", "Matrizes", "Filiais"],
).T

log.info("Etapa 5 concluída!")
log.info("\nIndicadores gerais:")
display(df_indicadores)